# Series Temporales en Python

Una **serie temporal** es una secuencia de observaciones ordenadas en el tiempo.

A diferencia de los datos tabulares tradicionales, aquí **el orden importa** y existe una **dependencia temporal** entre observaciones.

### Ejemplos en Business Analytics
*   **Energía**: Predecir demanda eléctrica para optimizar producción y evitar cortes.
*   **Retail**: Forecast de ventas para gestión de inventario.
*   **Finanzas**: Predicción de precios de acciones o divisas.
*   **Marketing**: Análisis de tráfico web y predicción de conversiones.
*   **Operaciones**: Previsión de demanda de servicios (call center, logística).

### Objetivo de la Sesión
Trabajaremos con datos reales de **consumo energético por hora** para aprender a:
1.  Entender los componentes de una serie temporal (tendencia, estacionalidad, ruido).
2.  Explorar patrones temporales (diarios, semanales).
3.  Usar **Prophet** (Facebook) para hacer predicciones robustas.
4.  Comparar con modelos baseline y estadísticos clásicos (ARIMA).
5.  Evaluar y validar modelos de forecasting.

**¿Por qué es importante?**
- Planificación de recursos (¿cuánta energía necesitaré mañana?).
- Detección de anomalías (¿es mi predicción muy distinta al consumo real?).
- Optimización de costes (¿cuándo es más barato comprar energía?).

In [ ]:
# Importación de librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Para análisis de series temporales
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Para Prophet (Facebook)
from prophet import Prophet

# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["date.autoformatter.hour"] = "%H:%M"

## 1. Carga y Exploración de Datos

Usaremos el dataset **Energy Consumption, Generation, Prices and Weather** de Kaggle, que contiene datos reales de España con:
- Consumo y generación energética por hora
- Precios de la electricidad

- Datos meteorológicosPara descargar los datos, necesitas tener configurada tu API key de Kaggle.


In [ ]:
# Cargamos el dataset de energía
df_raw = pd.read_csv("https://raw.githubusercontent.com/ami232/nebrija-mpan/main/data/energy_dataset.csv")

print(f"Dataset cargado: {len(df_raw):,} observaciones")
print(f"\nColumnas disponibles:")
print(df_raw.columns.tolist())
print(f"\nPrimeras filas:")
df_raw.head()

In [ ]:
# Procesamos los datos para nuestro análisis
# Nos centraremos en la carga total (total load actual)

# Convertimos la columna de tiempo a datetime
df_raw["time"] = pd.to_datetime(df_raw["time"], utc=True)

# Seleccionamos las columnas relevantes y renombramos
df = df_raw[["time", "total load actual"]].copy()
df = df.rename(columns={"time": "timestamp", "total load actual": "energy_mwh"})


# Ordenamos por tiempo
df = df.sort_values("timestamp").reset_index(drop=True)

# Info del dataset final
print(f"Dataset procesado: {len(df):,} observaciones")
print(f"Período: {df['timestamp'].min()} a {df['timestamp'].max()}")
print(f"\nEstadísticas de la Demanda Energética (MWh):")
print(df["energy_mwh"].describe())

# Nulos
print(f"\nNulos: {df.isnull().sum()}")
# Rellenamos cada nulo con el valor anterior observado
df = df.ffill()
print(f"\nNulos tras ffill: {df.isnull().sum()}")
df.head(10)

In [ ]:
# Visualización de la serie temporal completa
plt.figure(figsize=(16, 5))
plt.plot(df["timestamp"], df["energy_mwh"], linewidth=0.5, alpha=0.8)
plt.title("Demanda Energética Horaria - España", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización interactiva con Plotly (últimos 2 años)
import plotly.express as px

# Filtramos para mostrar solo los últimos 2 años
df_plot = df[df["timestamp"] >= df["timestamp"].max() - pd.DateOffset(years=2)]

fig = px.line(
    df_plot,
    x="timestamp",
    y="energy_mwh",
    title="Demanda Energética Horaria - España (Últimos 2 años)",
    labels={"timestamp": "Fecha", "energy_mwh": "Demanda (MWh)"},
    template="plotly_white",
    width=1000,
    height=600,
)
fig.update_layout(xaxis=dict(rangeslider=dict(visible=True)), font=dict(size=14))
fig.show()

## 2. Componentes de una Serie Temporal

Toda serie temporal se puede descomponer en cuatro componentes principales:

1. **Tendencia (Trend)**: Movimiento a largo plazo (crecimiento o decrecimiento).
2. **Estacionalidad (Seasonality)**: Patrones que se repiten en intervalos fijos (diarios, semanales, mensuales, anuales).
3. **Residuo/Ruido (Residual)**: Variación aleatoria que no podemos explicar con los componentes anteriores.

### Descomposición de la Serie

Usamos `seasonal_decompose()` de statsmodels para separar estos componentes automáticamente:

In [ ]:
# Preparamos los datos para descomposición
# Vamos a empezar con solo un año
# Necesitamos un índice temporal y una frecuencia regular

df_decompose = df[df.timestamp >= "2018-01-01"].set_index("timestamp")

# Descomponemos con estacionalidad semanal (24h * 7 días = 168 períodos)
decomposition = seasonal_decompose(
    df_decompose["energy_mwh"], model="additive", period=24 * 7
)

# Visualizamos los componentes
fig, axes = plt.subplots(4, 1, figsize=(16, 10))

# Serie original
decomposition.observed.plot(ax=axes[0], title="Serie Original", ylabel="MWh")
axes[0].grid(True, alpha=0.3)

# Tendencia
decomposition.trend.plot(ax=axes[1], title="Tendencia", ylabel="MWh", color="orange")
axes[1].grid(True, alpha=0.3)

# Estacionalidad
decomposition.seasonal.plot(
    ax=axes[2], title="Estacionalidad (Patrón Semanal)", ylabel="MWh", color="green"
)
axes[2].grid(True, alpha=0.3)

# Residuo
decomposition.resid.plot(
    ax=axes[3], title="Residuo (Ruido)", ylabel="MWh", color="red", alpha=0.7
)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Análisis Exploratorio de Patrones Temporales

Vamos a analizar los patrones de consumo en diferentes escalas temporales para entender el comportamiento del consumo.

In [ ]:
# Extraemos características temporales
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek  # 0=Lunes, 6=Domingo
df["month"] = df["timestamp"].dt.month
df["day_name"] = df["timestamp"].dt.day_name()

# Patrón por hora del día
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Consumo promedio por hora
hourly_avg = df.groupby("hour")["energy_mwh"].mean()
axes[0].bar(hourly_avg.index, hourly_avg.values, color="steelblue", alpha=0.7)
axes[0].set_title("Consumo Promedio por Hora del Día", fontweight="bold")
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("MWh Promedio")
axes[0].grid(axis="y", alpha=0.3)

# 2. Consumo promedio por día de la semana
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
weekly_avg = df.groupby("day_name")["energy_mwh"].mean().reindex(day_order)
axes[1].bar(range(7), weekly_avg.values, color="coral", alpha=0.7)
axes[1].set_title("Consumo Promedio por Día de la Semana", fontweight="bold")
axes[1].set_xlabel("Día")
axes[1].set_ylabel("MWh Promedio")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"], rotation=45)
axes[1].grid(axis="y", alpha=0.3)

# 3. Consumo promedio por mes
monthly_avg = df.groupby("month")["energy_mwh"].mean()
axes[2].bar(monthly_avg.index, monthly_avg.values, color="seagreen", alpha=0.7)
axes[2].set_title("Consumo Promedio por Mes", fontweight="bold")
axes[2].set_xlabel("Mes")
axes[2].set_ylabel("MWh Promedio")
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(
    [
        "Ene",
        "Feb",
        "Mar",
        "Abr",
        "May",
        "Jun",
        "Jul",
        "Ago",
        "Sep",
        "Oct",
        "Nov",
        "Dic",
    ],
    rotation=45,
)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Media Móvil (Rolling Average) - suaviza la serie para ver tendencias
df_plot = df.set_index("timestamp")

plt.figure(figsize=(16, 5))
plt.plot(
    df_plot.index, df_plot["energy_mwh"], label="Original", alpha=0.4, linewidth=0.5
)
plt.plot(
    df_plot.index,
    df_plot["energy_mwh"].rolling(window=24 * 7).mean(),
    label="Media Móvil 168h (1 semana)",
    color="red",
    linewidth=2,
)
plt.plot(
    df_plot.index,
    df_plot["energy_mwh"].rolling(window=24 * 30).mean(),
    label="Media Móvil 720h (1 mes)",
    color="green",
    linewidth=2,
)
plt.title("Serie Original vs Medias Móviles", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Las medias móviles suavizan el ruido y revelan tendencias")

## 4. Preparación de Datos para Modelado

En series temporales, **el orden importa**. No podemos mezclar aleatoriamente train y test como en ML tradicional.

### Train/Test Split Temporal
- **Train**: 21 meses (ej: Ene 2017 - Sep 2018)
- **Test**: Siguientes 3 meses (Oct-Dic 2018)

Esto simula la realidad: entrenar con datos históricos y predecir el futuro.

In [ ]:
# Dividimos temporalmente: primeros 21 meses para train, siguientes 3 para test
split_date = "2018-10-01"

train = df[df["timestamp"] < split_date].copy()
test = df[df["timestamp"] >= split_date].copy()

print(
    f"Train: {train['timestamp'].min()} a {train['timestamp'].max()} ({len(train):,} observaciones)"
)
print(
    f"Test:  {test['timestamp'].min()} a {test['timestamp'].max()} ({len(test):,} observaciones)"
)
print(
    f"Proporción: {len(train)/(len(train)+len(test))*100:.1f}% train, {len(test)/(len(train)+len(test))*100:.1f}% test"
)

# Visualizamos el split
plt.figure(figsize=(16, 4))
plt.plot(
    train["timestamp"], train["energy_mwh"], label="Train", alpha=0.7, linewidth=0.5
)
plt.plot(
    test["timestamp"],
    test["energy_mwh"],
    label="Test",
    color="red",
    alpha=0.7,
    linewidth=0.5,
)
plt.axvline(
    x=pd.to_datetime(split_date),
    color="black",
    linestyle="--",
    linewidth=2,
    label="Split",
)
plt.title("División Train/Test Temporal", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Modelos de Forecasting

### 5.1. Modelo Baseline: Naive Forecast

El modelo más simple: "La predicción de mañana es el valor de hoy".

En series temporales con estacionalidad fuerte, usamos **Seasonal Naive**: "La predicción de esta hora es la misma hora de la semana pasada".

In [ ]:
df["energy_mwh"].shift(7)

In [ ]:
# Seasonal Naive: predecimos con el valor de hace 1 semana (168 horas)
seasonal_lag = 24 * 7  # 1 semana

# Para cada punto del test, tomamos el valor de hace 1 semana del train/test
test["naive_pred"] = df["energy_mwh"].shift(seasonal_lag).loc[test.index].values


# Calculamos métricas
def calculate_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n{model_name}")
    print(f"  MAE:  {mae:.2f} MWh")
    print(f"  RMSE: {rmse:.2f} MWh")
    print(f"  MAPE: {mape:.2f}%")

    return {"model": model_name, "MAE": mae, "RMSE": rmse, "MAPE": mape}


results = []
results.append(
    calculate_metrics(test["energy_mwh"], test["naive_pred"], "Seasonal Naive")
)

# Visualizamos
plt.figure(figsize=(16, 5))
plt.plot(test["timestamp"], test["energy_mwh"], label="Real", alpha=0.8, linewidth=1.5)
plt.plot(
    test["timestamp"],
    test["naive_pred"],
    label="Seasonal Naive",
    alpha=0.7,
    linestyle="--",
    linewidth=1.5,
)
plt.title("Predicciones: Seasonal Naive", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Este modelo es nuestro baseline. Cualquier modelo más complejo debe superarlo.")

### 5.2. Prophet (Facebook)

**Prophet** es una librería de código abierto desarrollada por Meta (Facebook) diseñada para hacer forecasting de series temporales de manera robusta y con poco esfuerzo de ajuste.

#### ¿Por qué Prophet?
- **Fácil de usar**: Solo necesita dos columnas (`ds` para fechas, `y` para valores).
- **Maneja automáticamente**:
  - Múltiples estacionalidades (diaria, semanal, anual).
  - Días festivos y eventos especiales.
  - Valores faltantes.
  - Cambios de tendencia (changepoints).
- **Interpretable**: Descompone la predicción en componentes (tendencia + estacionalidades).

#### Fórmula de Prophet:
$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

Donde:
- $g(t)$: Tendencia (crecimiento lineal o logístico)
- $s(t)$: Estacionalidad (suma componentes periódicas)
- $h(t)$: Efectos de días festivos
- $\epsilon_t$: Ruido

In [ ]:
# Prophet requiere columnas específicas: 'ds' (datetime sin zona horaria) y 'y' (valor a predecir)
train_prophet = train[["timestamp", "energy_mwh"]].copy()
train_prophet["ds"] = train_prophet["timestamp"].dt.tz_localize(None)
train_prophet = train_prophet.rename(columns={"energy_mwh": "y"})
train_prophet = train_prophet[["ds", "y"]]

# Creamos y configuramos el modelo
model_prophet = Prophet(
    daily_seasonality=True,  # Estacionalidad diaria (24h)
    weekly_seasonality=True,  # Estacionalidad semanal
    yearly_seasonality=True,  # Estacionalidad anual
    seasonality_mode="additive",  # 'additive' o 'multiplicative'
    changepoint_prior_scale=0.05,  # Flexibilidad de cambios de tendencia (default=0.05)
    # holidays=...
)

# Entrenamos el modelo
print("Entrenando Prophet...")
model_prophet.fit(train_prophet)
print("Modelo entrenado")

In [ ]:
# Hacemos predicciones para el período de test
test_prophet = test[["timestamp"]].rename(columns={"timestamp": "ds"})
test_prophet["ds"] = test_prophet["ds"].dt.tz_localize(None)
forecast = model_prophet.predict(test_prophet)

# Extraemos las predicciones
test["prophet_pred"] = forecast["yhat"].values

# Calculamos métricas
results.append(calculate_metrics(test["energy_mwh"], test["prophet_pred"], "Prophet"))

# Visualizamos
plt.figure(figsize=(16, 5))
plt.plot(test["timestamp"], test["energy_mwh"], label="Real", alpha=0.8, linewidth=1.5)
plt.plot(
    test["timestamp"],
    test["prophet_pred"],
    label="Prophet",
    alpha=0.7,
    linestyle="--",
    linewidth=1.5,
)
plt.title("Predicciones: Prophet", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización interactiva con Plotly de las predicciones Prophet
import plotly.express as px

fig = px.line(
    test,
    x="timestamp",
    y=["energy_mwh", "prophet_pred"],
    labels={"value": "Demanda (MWh)", "timestamp": "Fecha", "variable": "Serie"},
    title="Predicción Prophet vs Realidad (Test)",
    template="plotly_white",
    width=1000,
    height=500,
    color_discrete_map={"energy_mwh": "#636EFA", "prophet_pred": "#EF553B"},
)
fig.update_traces(line=dict(width=2))
fig.show()

In [ ]:
# Visualización de componentes de Prophet
fig = model_prophet.plot_components(forecast)
plt.tight_layout()
plt.show()

print("Los componentes muestran:")
print("  - Trend: tendencia general creciente")
print("  - Weekly: patrón semanal (picos L-V, caídas S-D)")
print("  - Daily: patrón horario (picos en horas laborales)")
print("  - Yearly: estacionalidad anual (verano/invierno)")

### 5.3. SARIMA (Seasonal ARIMA)

**ARIMA** (AutoRegressive Integrated Moving Average) es el modelo estadístico clásico para series temporales.

#### Componentes:
- **AR (p)**: AutoRegresivo - usa valores pasados de la serie ($y_{t-1}, y_{t-2}, ...$)
- **I (d)**: Integrado - diferenciación para hacer la serie estacionaria
- **MA (q)**: Media Móvil - usa errores pasados del modelo

**SARIMA** añade componentes estacionales: $(p,d,q) \times (P,D,Q)_s$

Para simplificar, usaremos parámetros estándar. En la práctica, se seleccionan con técnicas como ACF/PACF o auto_arima.

In [ ]:
# SARIMA con estacionalidad semanal (24*7=168)
# Usamos parámetros simples: (1,1,1) x (1,1,1,168)
# Nota: entrenar SARIMA con muchos datos puede ser lento

print("Entrenando SARIMA... (esto puede tardar unos minutos)")

# Para acelerar, usamos solo una muestra del train
train_sarima = train.tail(24 * 42)  # Últimos 42 días

model_sarima = SARIMAX(
    train_sarima["energy_mwh"],
    order=(1, 1, 1),  # (p, d, q)
    seasonal_order=(1, 1, 1, 168),  # (P, D, Q, s) - estacionalidad semanal
    enforce_stationarity=False,
    enforce_invertibility=False,
)

results_sarima = model_sarima.fit(disp=False)
print("✓ Modelo SARIMA entrenado")

# Predicciones
sarima_forecast = results_sarima.forecast(steps=len(test))
test["sarima_pred"] = sarima_forecast.values

# Métricas
results.append(calculate_metrics(test["energy_mwh"], test["sarima_pred"], "SARIMA"))

# Visualización
plt.figure(figsize=(16, 5))
plt.plot(test["timestamp"], test["energy_mwh"], label="Real", alpha=0.8, linewidth=1.5)
plt.plot(
    test["timestamp"],
    test["sarima_pred"],
    label="SARIMA",
    alpha=0.7,
    linestyle="--",
    linewidth=1.5,
)
plt.title("Predicciones: SARIMA", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Comparación de Modelos

Comparamos todos los modelos usando las métricas calculadas:

In [ ]:
# Tabla comparativa
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("RMSE")

print("\n" + "=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)
display(
    df_results.style.highlight_min(subset=["MAE", "RMSE", "MAPE"], color="lightgreen")
)

# Gráfico de barras
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, metric in enumerate(["MAE", "RMSE", "MAPE"]):
    axes[idx].barh(
        df_results["model"], df_results[metric], color="steelblue", alpha=0.7
    )
    axes[idx].set_xlabel(metric)
    axes[idx].set_title(f"{metric} por Modelo", fontweight="bold")
    axes[idx].grid(axis="x", alpha=0.3)
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nConclusiones:")
print("  - Prophet supera claramente al baseline Naive")
print("  - SARIMA es competitivo pero más complejo de configurar")
print("  - Para producción, Prophet es ideal por su facilidad de uso y robustez")

In [ ]:
# Comparación visual de todas las predicciones
plt.figure(figsize=(16, 6))

# Solo mostramos 2 semanas para ver detalles
test_subset = test.head(24 * 14)  # Primeras 2 semanas

plt.plot(
    test_subset["timestamp"],
    test_subset["energy_mwh"],
    label="Real",
    color="black",
    linewidth=2,
    alpha=0.8,
)
plt.plot(
    test_subset["timestamp"],
    test_subset["naive_pred"],
    label="Seasonal Naive",
    linestyle=":",
    linewidth=2,
    alpha=0.7,
)
plt.plot(
    test_subset["timestamp"],
    test_subset["prophet_pred"],
    label="Prophet",
    linestyle="--",
    linewidth=2,
    alpha=0.7,
)
plt.plot(
    test_subset["timestamp"],
    test_subset["sarima_pred"],
    label="SARIMA",
    linestyle="-.",
    linewidth=2,
    alpha=0.7,
)

plt.title(
    "Comparación de Modelos (Primeras 2 Semanas de Test)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Predicción Futura con Prophet

Una vez validado el modelo, podemos usarlo para predecir el futuro (ej: próximas 2 semanas).

In [ ]:
# Entrenamos Prophet con TODOS los datos (train + test) para predecir el futuro real
df_full_prophet = df[["timestamp", "energy_mwh"]].rename(
    columns={"timestamp": "ds", "energy_mwh": "y"}
)

model_prophet_final = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    seasonality_mode="additive",
)

model_prophet_final.fit(df_full_prophet)

# Creamos fechas futuras: próximas 2 semanas (336 horas)
future = model_prophet_final.make_future_dataframe(periods=24 * 14, freq="H")

# Predecimos
forecast_future = model_prophet_final.predict(future)

# Visualizamos
fig = model_prophet_final.plot(forecast_future, figsize=(16, 6))
plt.title(
    "Predicción de Demanda Energética - Próximas 2 Semanas",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Fecha")
plt.ylabel("Demanda (MWh)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Las bandas azules representan el intervalo de confianza (80%)")
print("Podemos usar estas predicciones para planificar recursos y negociar precios")

In [ ]:
# Zoom en las predicciones futuras (solo lo que viene después de nuestros datos)
forecast_only_future = forecast_future[forecast_future["ds"] > df["timestamp"].max()]

plt.figure(figsize=(16, 5))
plt.plot(
    forecast_only_future["ds"],
    forecast_only_future["yhat"],
    label="Predicción",
    color="blue",
    linewidth=2,
)
plt.fill_between(
    forecast_only_future["ds"],
    forecast_only_future["yhat_lower"],
    forecast_only_future["yhat_upper"],
    alpha=0.3,
    color="blue",
    label="Intervalo de confianza 80%",
)
plt.title("Forecast: Próximas 2 Semanas (Enero 2025)", fontsize=14, fontweight="bold")
plt.xlabel("Fecha")
plt.ylabel("Demanda Predicha (MWh)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Predicción para las próximas {len(forecast_only_future)} horas")
print(f"Demanda promedio esperada: {forecast_only_future['yhat'].mean():.0f} MWh")
print(f"Pico máximo esperado: {forecast_only_future['yhat'].max():.0f} MWh")
print(f"Valle mínimo esperado: {forecast_only_future['yhat'].min():.0f} MWh")

## 8. Conceptos Clave y Buenas Prácticas

### Diferencias con ML Tradicional
| Aspecto | ML Tradicional | Series Temporales |
|---------|----------------|-------------------|
| **Train/Test Split** | Aleatorio | Temporal (pasado→futuro) |
| **Features** | Independientes | Dependientes del tiempo |
| **Validación** | K-Fold CV | Time Series CV (walk-forward) |
| **Objetivo** | Clasificar/predecir cualquier punto | Predecir el futuro |

### Métricas Comunes
- **MAE** (Mean Absolute Error): Error promedio en las mismas unidades que la variable.
- **RMSE** (Root Mean Squared Error): Penaliza más los errores grandes.
- **MAPE** (Mean Absolute Percentage Error): Error porcentual, útil para comparar series en distintas escalas.

### Cuándo usar cada modelo
- **Seasonal Naive**: Baseline rápido, útil para datos muy estacionales.
- **Prophet**: Primera opción para la mayoría de casos. Fácil, robusto, interpreta múltiples estacionalidades.
- **ARIMA/SARIMA**: Cuando necesitas control total y entiendes los parámetros. Mejor para series estacionarias.
- **ML (Random Forest, XGBoost)**: Cuando tienes muchas features exógenas (temperatura, festivos, promociones) y quieres capturar relaciones no lineales.

### Validación Temporal (Time Series CV)
En producción, deberías validar con **walk-forward validation**:
1. Entrenar con datos hasta el mes 12
2. Predecir el mes 13
3. Agregar mes 13 al train
4. Predecir mes 14
5. Repetir...

Esto simula cómo funcionará el modelo en producción (entrenamiento continuo).

---
## Ejercicios para Clase

### Ejercicio 1: Análisis de Residuos
Los residuos (errores del modelo) deberían ser ruido blanco (aleatorio, sin patrones).

1. Calcula los residuos de Prophet: `test['residuos'] = test['energy_mwh'] - test['prophet_pred']`
2. Crea un gráfico de línea de los residuos en el tiempo
3. Crea un histograma de los residuos
4. ¿Observas algún patrón? ¿Hay sesgo (media ≠ 0)? ¿Parece normal la distribución?

In [ ]:
# TU CÓDIGO AQUÍ - Ejercicio 1

### Ejercicio 2: Ajustar Prophet con Estacionalidades Personalizadas

Prophet permite añadir estacionalidades personalizadas. Vuelve a entrenar el modelo pero esta vez:

1. Desactiva la estacionalidad diaria automática: `daily_seasonality=False`
2. Añade una estacionalidad diaria personalizada con más Fourier terms (más componentes periódicas):
   ```python
   model.add_seasonality(name='hourly', period=1, fourier_order=15)
   ```
3. Entrena el modelo y compara las métricas con el Prophet original.
4. ¿Mejora el RMSE? ¿Por qué sí o por qué no?

In [ ]:
# TU CÓDIGO AQUÍ - Ejercicio 2

### Ejercicio 3: Predicción con Variables Exógenas

La demanda energética depende de factores externos como la temperatura.

1. Usa el otro archivo descargado que incluye la temperatura y añádelo al dataset (promedia las temperaturas de las ciudades en cada hora)
2. En Prophet, puedes añadir regresores (variables adicionales):
   ```python
   model.add_regressor('temp')
   ```
3. Entrena un nuevo modelo Prophet con esta variable y evalúa si mejora las predicciones.
4. **Pista**: Tendrás que incluir la columna `temp` tanto en los datos de entrenamiento como en el dataframe de predicción (test/future).

**Reflexión**: ¿Por qué es importante incluir variables exógenas en series temporales de negocio?

In [ ]:
# TU CÓDIGO AQUÍ - Ejercicio 3